# Equal risk contribution: 50 CSI 300 names

This notebook does two things. First, it builds equal-risk-contribution weights for the 50 largest index names from the sample covariance over October 2021 through September 2026, and checks that each name's risk contribution is close to 1/50. Second, it compares those weights with rescaled index weights over the 17 trading days in October 2026. One month is not a track record.


# 风险平价：50 只沪深 300 成分股

这个笔记本做两件事。第一，用 2021-10 到 2026-09 的样本协方差，给指数权重最大的 50 只股票求风险平价权重，并检查每只股票的风险贡献是否接近 1/50。第二，用 2026 年 10 月的 17 个交易日，把这组权重和重新归一的指数权重放在一起看。对照只有这一个月，不能当成业绩记录。


In [1]:
import pandas as pd
import numpy as np
import edhec_risk_kit as erk


## Data and universe

The weights are the cached row for 2026-09-30, the second-to-last row. The last row is 2026-10-30, and using it for an October comparison would take weights that were not known until the end of that month. The estimation window is 2021-10-01 through 2026-09-30. Names missing more than 5% of that window are dropped, the 50 largest remaining index weights are kept, and those 50 weights are rescaled to sum to 1. This is a large-weight subset, not the full CSI 300.


## 数据和股票池

权重用缓存表的 2026-09-30 那一行，也就是倒数第二行。最后一行是 2026-10-30；拿它来做 10 月的对照，会用到月末才公布的权重。估计区间是 2021-10-01 到 2026-09-30。先去掉这个区间里缺失超过 5% 的股票，再留下指数权重最大的 50 只，并把这 50 只的指数权重重新加总为 1。这是大权重股票的一个子集，不是完整的沪深 300。


In [2]:
snapshot = erk.read_hs300_weights().iloc[-2].dropna().sort_values(ascending=False)
raw = erk.read_returns()
holdout = raw.loc["2026-10-01":"2026-10-31"]
history = raw.loc["2021-10-01":"2026-09-30"]
history = history.loc[:, history.isna().mean() <= 0.05]
codes = snapshot.index[snapshot.index.isin(history.columns)][:50]
snapshot = snapshot.loc[codes]
snapshot = snapshot / snapshot.sum()
history = history[codes]
holdout = holdout[codes]
print(snapshot.name.date(), history.index.min().date(), history.index.max().date(), len(holdout), len(codes))


2026-09-30 2021-10-08 2026-09-30 17 50


## Risk contribution

Name i contributes `w_i * (Σw)_i / (w'Σw)` of portfolio variance. `equal_risk_contributions` takes the sample covariance above and, with long-only weights that sum to 1, pulls each contribution toward 1/50. The solver is SLSQP in `edhec_risk_kit.py`. The cell prints every contribution and the largest absolute gap from 1/50.

In the next cell the largest absolute gap from 1/50 is about 0.0004.


## 风险贡献

第 i 只股票的风险贡献是 `w_i * (Σw)_i / (w'Σw)`。`equal_risk_contributions` 用上面的样本协方差，在只做多、权重和为 1 的约束下，把每只股票的贡献拉向 1/50。求解在 `edhec_risk_kit.py` 里，方法是 SLSQP。下面给出每只股票的贡献，以及它们离 1/50 的最大绝对偏差。

下面这格里，贡献离 1/50 的最大绝对偏差大约是 0.0004。


In [3]:
cov = history.cov()
erc = pd.Series(erk.equal_risk_contributions(cov), index=cov.index)
contribution = pd.Series(np.asarray(erk.risk_contribution(erc, cov), dtype=float), index=cov.index)
target = 1 / len(contribution)
print(float((contribution - target).abs().max()))
contribution.sort_values()


0.0003795163117247327


601398.SH    0.019620
601088.SH    0.019796
300059.SZ    0.019863
600000.SH    0.019866
603993.SH    0.019870
601816.SH    0.019880
603019.SH    0.019892
600919.SH    0.019899
601318.SH    0.019917
688012.SH    0.019942
002415.SZ    0.019945
300476.SZ    0.019952
002594.SZ    0.019954
600900.SH    0.019965
300502.SZ    0.019973
601138.SH    0.019974
000858.SZ    0.019974
688008.SH    0.019974
600030.SH    0.019977
300274.SZ    0.019979
000725.SZ    0.019980
300308.SZ    0.019984
002371.SZ    0.019987
688256.SH    0.019989
601899.SH    0.019992
688981.SH    0.019997
300394.SZ    0.020000
300750.SZ    0.020002
600183.SH    0.020002
603986.SH    0.020006
002384.SZ    0.020007
603259.SH    0.020032
601857.SH    0.020033
000338.SZ    0.020036
600276.SH    0.020042
601211.SH    0.020050
300408.SZ    0.020052
600887.SH    0.020060
600519.SH    0.020064
000651.SZ    0.020067
000333.SZ    0.020069
002475.SZ    0.020070
600150.SH    0.020072
601166.SH    0.020085
600309.SH    0.020091
601288.SH 

## One-month comparison

The October 2026 return compounds daily weighted simple returns. The denominator is not the realized volatility of those 17 days. It is the square root of the estimation-window daily covariance multiplied by 17, which is the holding-period model volatility if daily returns are iid. The cash return scales a 2% annual rate over `17/252`. The ratio in the table is (realized return − cash return) / model volatility. The numerator is an outcome and the denominator is a forecast, so the ratio is not a Sharpe ratio.

Over these 17 days the rescaled index weights have a realized return of about -11.8% and a model volatility of about 5.7%, so the ratio is about -2.10. Equal-risk weights have a realized return of about -3.7% and a model volatility of about 3.9%, so the ratio is about -0.96.


## 一个月的对照

2026-10 的收益是日简单收益按权重加总后再复利。分母不是这 17 天的已实现波动，而是估计区间的日协方差乘以 17 再开方：如果日收益独立同分布，这就是这段持有期的模型波动。无风险收益按年化 2% 折到 `17/252`。表里的比值是（实现收益 − 同期无风险收益）/ 模型波动。分子是事后收益，分母是事前波动，所以它不是夏普比率。

重新归一的指数权重在这 17 天的实现收益大约是 -11.8%，模型波动大约是 5.7%，比值大约是 -2.10。风险平价的实现收益大约是 -3.7%，模型波动大约是 3.9%，比值大约是 -0.96。


In [4]:
n_days = len(holdout)
holding_cov = cov * n_days
cash = (1 + 0.02) ** (n_days / 252) - 1

def realized_return(weight):
    gross = (holdout + 1) @ weight
    return float(gross.prod() - 1)

def model_volatility(weight):
    variance = float(weight @ holding_cov @ weight)
    return variance ** 0.5

rows = []
for name, weight in [("rescaled index", snapshot), ("equal risk", erc)]:
    ret = realized_return(weight)
    vol = model_volatility(weight)
    rows.append({
        "sleeve": name,
        "realized_return": ret,
        "model_volatility": vol,
        "excess_over_model_volatility": (ret - cash) / vol,
    })
pd.DataFrame(rows).set_index("sleeve")


,realized_return,model_volatility,excess_over_model_volatility
sleeve,,,
rescaled index,-0.117549,0.056502,-2.104117
equal risk,-0.036516,0.039340,-0.962206
